<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V1 Dynamic Re-centering Grid + Risk Guardrails

V1 develops the frozen V0-B baseline in only two strategy areas:

1. **Dynamic re-centering** — the active 30-grid range moves with the market.
2. **Risk guardrails** — new BUY entries are blocked when portfolio risk limits are reached.

V0-B execution semantics are retained: initialized spot inventory, fixed arithmetic gap, fixed order size, SELL before BUY, downward-crossing BUYs, no same-candle rebuy, same-candle SELL proceeds cannot fund BUYs, and no compounding.

Old positions survive re-centering and keep their original SELL targets.

This notebook is for **backtest / shadow-readiness only**. It does not send Binance orders.


# 0. Setup


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import base64
import bisect
import heapq
import json
import os
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests


# 1. Trading System

The initial active range is user-defined. With the current settings:

- Initial reference = 42,000 USDT
- 30 grids
- Gap = 1,000 USDT
- Initial floor / ceiling = 27,000 / 57,000 USDT
- Re-center trigger = ±5 grids = ±5,000 USDT

After re-centering, the new active range remains `Reference ±15,000`.


## 1.1 Trading Configuration


In [ ]:
SYMBOL = "BTCUSDT"
INITIAL_CAPITAL = 3000.0

INITIAL_FLOOR = 27000.0
INITIAL_CEILING = 57000.0
GRID_GAP = 1000.0

BUY_FEE = 0.001
SELL_FEE = 0.001

RECENTER_TRIGGER_GRIDS = 5

# Provisional V1 test guardrails for NEW BUY entries.
# Keep these explicit so we can review/tune them after the first V1 result.
MIN_CASH_RESERVE = 750.0
MAX_OPEN_POSITIONS = 18
MAX_DEPLOYED_CAPITAL = 1800.0
MAX_ENTRY_BTC_EXPOSURE = 1800.0
MAX_DRAWDOWN_STOP = 0.15

LIVE_EXECUTION_ENABLED = False
EXECUTION_MODE = "BACKTEST / SHADOW READINESS"


## 1.2 Grid, Re-centering & Initialized Portfolio


In [ ]:
def validate_v1_config(
    capital,
    floor,
    ceiling,
    gap,
    buy_fee,
    sell_fee,
    recenter_trigger_grids,
    min_cash_reserve,
    max_open_positions,
    max_deployed_capital,
    max_entry_btc_exposure,
    max_drawdown_stop,
):
    if capital <= 0 or floor <= 0 or ceiling <= floor or gap <= 0:
        raise ValueError("Invalid grid configuration.")

    if not (0 <= buy_fee < 1 and 0 <= sell_fee < 1):
        raise ValueError("Fees must be in [0, 1).")

    raw_count = (ceiling - floor) / gap
    if not np.isclose(raw_count, round(raw_count)):
        raise ValueError("Initial range must be exactly divisible by GRID_GAP.")

    number_of_grids = int(round(raw_count))
    if number_of_grids < 2:
        raise ValueError("At least two grids are required.")

    initial_reference = (floor + ceiling) / 2.0
    if not np.isclose(initial_reference / gap, round(initial_reference / gap)):
        raise ValueError("Initial reference midpoint must align with GRID_GAP.")

    if recenter_trigger_grids <= 0:
        raise ValueError("RECENTER_TRIGGER_GRIDS must be > 0.")

    if not (0 <= min_cash_reserve < capital):
        raise ValueError("MIN_CASH_RESERVE must be within [0, INITIAL_CAPITAL).")

    if max_open_positions is not None and max_open_positions <= 0:
        raise ValueError("MAX_OPEN_POSITIONS must be > 0 or None.")

    if max_deployed_capital is not None and max_deployed_capital <= 0:
        raise ValueError("MAX_DEPLOYED_CAPITAL must be > 0 or None.")

    if max_entry_btc_exposure is not None and max_entry_btc_exposure <= 0:
        raise ValueError("MAX_ENTRY_BTC_EXPOSURE must be > 0 or None.")

    if max_drawdown_stop is not None and not (0 < max_drawdown_stop < 1):
        raise ValueError("MAX_DRAWDOWN_STOP must be between 0 and 1 or None.")

    return number_of_grids, float(initial_reference)


NUMBER_OF_GRIDS, INITIAL_REFERENCE = validate_v1_config(
    INITIAL_CAPITAL,
    INITIAL_FLOOR,
    INITIAL_CEILING,
    GRID_GAP,
    BUY_FEE,
    SELL_FEE,
    RECENTER_TRIGGER_GRIDS,
    MIN_CASH_RESERVE,
    MAX_OPEN_POSITIONS,
    MAX_DEPLOYED_CAPITAL,
    MAX_ENTRY_BTC_EXPOSURE,
    MAX_DRAWDOWN_STOP,
)


def round_to_gap(price, gap):
    return float(np.floor(float(price) / gap + 0.5) * gap)


def build_regime(reference_price, gap, number_of_grids, regime_id):
    lower_grids = number_of_grids // 2
    upper_grids = number_of_grids - lower_grids

    floor = reference_price - lower_grids * gap
    ceiling = reference_price + upper_grids * gap

    if floor <= 0:
        raise ValueError("Dynamic floor must stay above zero.")

    buy_prices = np.arange(floor, ceiling, gap, dtype=float)
    if len(buy_prices) != number_of_grids:
        raise AssertionError("Dynamic regime grid count mismatch.")

    return {
        "regime_id": int(regime_id),
        "reference_price": float(reference_price),
        "floor": float(floor),
        "ceiling": float(ceiling),
        "buy_prices": buy_prices,
        "sell_targets": buy_prices + gap,
    }


def derive_initialized_regime(regime, start_price, initial_capital, buy_fee):
    buy_prices = regime["buy_prices"]
    sell_targets = regime["sell_targets"]

    if not (regime["floor"] < start_price < regime["ceiling"]):
        raise ValueError(
            "Starting market price must be inside the initial V1 range. "
            "Adjust INITIAL_FLOOR / INITIAL_CEILING before running."
        )

    # Same V0-B initialization principle:
    # slots whose SELL target is above the starting market are seeded with BTC.
    seed_mask = sell_targets > start_price
    reserve_mask = ~seed_mask
    seed_buy_prices = buy_prices[seed_mask]

    funding_weight = (
        int(reserve_mask.sum())
        + float(np.sum(start_price / seed_buy_prices))
    )
    normal_order_size = initial_capital / funding_weight

    normal_net_btc = normal_order_size / buy_prices * (1.0 - buy_fee)
    initial_entry_costs = np.where(
        seed_mask,
        normal_net_btc / (1.0 - buy_fee) * start_price,
        0.0,
    )

    reserved_cash = float(reserve_mask.sum() * normal_order_size)
    seeded_btc_cost = float(initial_entry_costs.sum())

    if not np.isclose(
        reserved_cash + seeded_btc_cost,
        initial_capital,
        atol=1e-8,
    ):
        raise AssertionError("V1 initialization does not reconcile to capital.")

    grid = pd.DataFrame({
        "grid_slot": np.arange(1, len(buy_prices) + 1),
        "grid_buy_price": buy_prices,
        "sell_target": sell_targets,
        "seed_at_start": seed_mask,
        "normal_net_btc": normal_net_btc,
        "initial_entry_cost_usdt": initial_entry_costs,
    })

    return grid, {
        "normal_order_size_usdt": float(normal_order_size),
        "funding_weight": float(funding_weight),
        "initial_sell_positions": int(seed_mask.sum()),
        "initial_buy_levels": int(reserve_mask.sum()),
        "reserved_cash_usdt": reserved_cash,
        "seeded_btc_cost_usdt": seeded_btc_cost,
    }


INITIAL_REGIME = build_regime(
    INITIAL_REFERENCE,
    GRID_GAP,
    NUMBER_OF_GRIDS,
    regime_id=0,
)

print(f"Initial Reference : {INITIAL_REFERENCE:,.0f}")
print(f"Initial Range     : {INITIAL_REGIME['floor']:,.0f} - {INITIAL_REGIME['ceiling']:,.0f}")
print(f"Number of Grids   : {NUMBER_OF_GRIDS}")
print(f"Grid Gap          : {GRID_GAP:,.0f}")
print(f"Recenter Trigger  : ±{RECENTER_TRIGGER_GRIDS * GRID_GAP:,.0f}")


## 1.3 Position & Dynamic Grid Engine


In [ ]:
def create_position(
    trade_id,
    regime_id,
    reference_price,
    buy_time,
    market_buy_price,
    grid_buy_price,
    sell_target,
    normal_order_size,
    portfolio_value_at_buy,
    buy_fee,
    sell_fee,
    entry_type,
):
    if entry_type == "INITIAL_SEED":
        # Match the BTC quantity that a normal future BUY at this logical
        # grid level would create, but acquire it at the actual start market.
        target_net_btc = (
            normal_order_size / grid_buy_price * (1.0 - buy_fee)
        )
        gross_btc = target_net_btc / (1.0 - buy_fee)
        order_size_usdt = gross_btc * market_buy_price
    else:
        order_size_usdt = normal_order_size
        gross_btc = order_size_usdt / market_buy_price

    buy_fee_btc = gross_btc * buy_fee
    btc_amount = gross_btc - buy_fee_btc
    gross_sell_usdt = btc_amount * sell_target
    sell_fee_usdt = gross_sell_usdt * sell_fee

    return {
        "trade_id": int(trade_id),
        "regime_id": int(regime_id),
        "reference_price_at_buy": float(reference_price),
        "entry_type": entry_type,
        "status": "OPEN",
        "buy_time": buy_time,
        "buy_price": float(market_buy_price),
        "grid_buy_price": float(grid_buy_price),
        "sell_target": float(sell_target),
        "sell_time": pd.NaT,
        "order_size_usdt": float(order_size_usdt),
        "portfolio_value_at_buy": float(portfolio_value_at_buy),
        "order_pct_of_portfolio": float(
            order_size_usdt / portfolio_value_at_buy
        ),
        "portfolio_value_at_sell": np.nan,
        "btc_amount": float(btc_amount),
        "buy_fee_btc": float(buy_fee_btc),
        "sell_fee_usdt": float(sell_fee_usdt),
        "net_sell_usdt": float(gross_sell_usdt - sell_fee_usdt),
        "net_pnl": np.nan,
    }


def initialize_v1_portfolio(
    data,
    initial_regime,
    initial_capital,
    buy_fee,
    sell_fee,
    min_cash_reserve,
    max_open_positions,
    max_deployed_capital,
    max_entry_btc_exposure,
):
    start_time = data.iloc[0]["open_time"]
    start_price = float(data.iloc[0]["open"])

    initial_grid, sizing = derive_initialized_regime(
        initial_regime,
        start_price,
        initial_capital,
        buy_fee,
    )

    positions = {}
    open_by_grid_price = {}
    sell_heap = []
    events = []

    cash = float(initial_capital)
    btc = 0.0
    deployed_capital = 0.0
    total_buy_fee_usdt = 0.0
    trade_id = 0
    event_id = 0

    for row in initial_grid.loc[initial_grid["seed_at_start"]].itertuples(index=False):
        portfolio_value = cash + btc * start_price
        trade_id += 1

        position = create_position(
            trade_id=trade_id,
            regime_id=initial_regime["regime_id"],
            reference_price=initial_regime["reference_price"],
            buy_time=start_time,
            market_buy_price=start_price,
            grid_buy_price=float(row.grid_buy_price),
            sell_target=float(row.sell_target),
            normal_order_size=float(sizing["normal_order_size_usdt"]),
            portfolio_value_at_buy=portfolio_value,
            buy_fee=buy_fee,
            sell_fee=sell_fee,
            entry_type="INITIAL_SEED",
        )

        cash_before, btc_before = cash, btc

        cash -= position["order_size_usdt"]
        btc += position["btc_amount"]
        deployed_capital += position["order_size_usdt"]
        total_buy_fee_usdt += position["buy_fee_btc"] * start_price

        positions[trade_id] = position
        open_by_grid_price[position["grid_buy_price"]] = trade_id
        heapq.heappush(sell_heap, (position["sell_target"], trade_id))

        event_id += 1
        events.append({
            "event_id": event_id,
            "time": start_time,
            "side": "BUY",
            "trade_id": trade_id,
            "regime_id": initial_regime["regime_id"],
            "grid_buy_price": position["grid_buy_price"],
            "price": start_price,
            "cash_movement": -position["order_size_usdt"],
            "grid_cashflow": 0.0,
            "cash_before": cash_before,
            "cash_after": cash,
            "btc_before": btc_before,
            "btc_after": btc,
            "deployed_capital_after": deployed_capital,
            "entry_exposure_after": btc * start_price,
            "initialization_trade": True,
        })

    initial_exposure = btc * start_price

    if cash < min_cash_reserve - 1e-9:
        raise ValueError(
            "Initial V1 allocation violates MIN_CASH_RESERVE. "
            "Adjust the initial range or risk limit."
        )
    if max_open_positions is not None and len(open_by_grid_price) > max_open_positions:
        raise ValueError(
            "Initial V1 allocation violates MAX_OPEN_POSITIONS."
        )
    if max_deployed_capital is not None and deployed_capital > max_deployed_capital + 1e-9:
        raise ValueError(
            "Initial V1 allocation violates MAX_DEPLOYED_CAPITAL."
        )
    if max_entry_btc_exposure is not None and initial_exposure > max_entry_btc_exposure + 1e-9:
        raise ValueError(
            "Initial V1 allocation violates MAX_ENTRY_BTC_EXPOSURE."
        )

    initialization = {
        **sizing,
        "start_time": start_time,
        "start_price": start_price,
        "initial_cash": float(cash),
        "initial_btc": float(btc),
        "initial_btc_cost_usdt": float(deployed_capital),
        "initial_buy_fee_usdt": float(total_buy_fee_usdt),
        "initial_btc_allocation_pct": float(
            deployed_capital / initial_capital * 100.0
        ),
        "initial_btc_market_value_usdt": float(initial_exposure),
    }

    return {
        "initial_grid": initial_grid,
        "cash": cash,
        "btc": btc,
        "deployed_capital": deployed_capital,
        "positions": positions,
        "open_by_grid_price": open_by_grid_price,
        "sell_heap": sell_heap,
        "events": events,
        "trade_id": trade_id,
        "event_id": event_id,
        "initialization": initialization,
        "total_buy_fee_usdt": total_buy_fee_usdt,
    }


def run_dynamic_recenter_grid(
    data,
    initial_capital,
    initial_reference,
    gap,
    number_of_grids,
    recenter_trigger_grids,
    buy_fee,
    sell_fee,
    min_cash_reserve,
    max_open_positions,
    max_deployed_capital,
    max_entry_btc_exposure,
    max_drawdown_stop,
):
    data = data.sort_values("open_time").reset_index(drop=True).copy()
    if data.empty:
        raise ValueError("Market data is empty.")

    regime_id = 0
    regime = build_regime(
        initial_reference,
        gap,
        number_of_grids,
        regime_id,
    )

    state = initialize_v1_portfolio(
        data=data,
        initial_regime=regime,
        initial_capital=initial_capital,
        buy_fee=buy_fee,
        sell_fee=sell_fee,
        min_cash_reserve=min_cash_reserve,
        max_open_positions=max_open_positions,
        max_deployed_capital=max_deployed_capital,
        max_entry_btc_exposure=max_entry_btc_exposure,
    )

    cash = state["cash"]
    btc = state["btc"]
    deployed_capital = state["deployed_capital"]
    positions = state["positions"]
    open_by_grid_price = state["open_by_grid_price"]
    sell_heap = state["sell_heap"]
    events = state["events"]
    trade_id = state["trade_id"]
    event_id = state["event_id"]
    initialization = state["initialization"]

    normal_order_size = float(initialization["normal_order_size_usdt"])
    total_buy_fee_usdt = float(state["total_buy_fee_usdt"])
    total_sell_fee_usdt = 0.0
    realized_profit = 0.0
    completed_cycles = 0

    regime_history = [{
        "regime_id": 0,
        "effective_time": data.iloc[0]["open_time"],
        "reference_price": regime["reference_price"],
        "floor": regime["floor"],
        "ceiling": regime["ceiling"],
        "reason": "INITIAL",
    }]
    recenter_events = []
    risk_halt_events = []

    blocked = {
        "risk_halt": 0,
        "cash_reserve": 0,
        "max_open_positions": 0,
        "max_deployed_capital": 0,
        "max_entry_btc_exposure": 0,
    }

    n = len(data)
    equity_values = np.empty(n)
    cash_values = np.empty(n)
    btc_values = np.empty(n)
    deployed_values = np.empty(n)
    open_position_values = np.empty(n, dtype=int)
    btc_market_value_values = np.empty(n)
    reference_values = np.empty(n)
    regime_values = np.empty(n, dtype=int)
    risk_halt_values = np.empty(n, dtype=bool)

    previous_close = None
    risk_halt = False
    peak_equity = float(initial_capital)
    tolerance = 1e-12

    for i, candle in enumerate(data.itertuples(index=False)):
        timestamp = candle.open_time
        open_price = float(candle.open)
        high_price = float(candle.high)
        low_price = float(candle.low)
        close_price = float(candle.close)

        # IMPORTANT: same-candle SELL proceeds cannot fund BUYs.
        cash_at_candle_start = cash
        buy_budget = cash_at_candle_start
        sold_this_candle = set()

        # 1) SELL existing positions first.
        while sell_heap and sell_heap[0][0] <= high_price + tolerance:
            _, current_trade_id = heapq.heappop(sell_heap)
            position = positions.get(current_trade_id)

            if position is None or position["status"] != "OPEN":
                continue

            grid_buy_price = position["grid_buy_price"]
            cash_before, btc_before = cash, btc
            portfolio_value = (
                cash_before + btc_before * position["sell_target"]
            )

            cash += position["net_sell_usdt"]
            btc -= position["btc_amount"]
            deployed_capital -= position["order_size_usdt"]

            if abs(btc) < 1e-12:
                btc = 0.0
            if abs(deployed_capital) < 1e-10:
                deployed_capital = 0.0

            pnl = (
                position["net_sell_usdt"]
                - position["order_size_usdt"]
            )
            position.update(
                status="CLOSED",
                sell_time=timestamp,
                portfolio_value_at_sell=portfolio_value,
                net_pnl=float(pnl),
            )

            open_by_grid_price.pop(grid_buy_price, None)
            sold_this_candle.add(grid_buy_price)

            total_sell_fee_usdt += position["sell_fee_usdt"]
            realized_profit += pnl
            completed_cycles += 1

            event_id += 1
            events.append({
                "event_id": event_id,
                "time": timestamp,
                "side": "SELL",
                "trade_id": current_trade_id,
                "regime_id": position["regime_id"],
                "grid_buy_price": grid_buy_price,
                "price": position["sell_target"],
                "cash_movement": position["net_sell_usdt"],
                "grid_cashflow": pnl,
                "cash_before": cash_before,
                "cash_after": cash,
                "btc_before": btc_before,
                "btc_after": btc,
                "deployed_capital_after": deployed_capital,
                "initialization_trade": False,
            })

        # 2) BUY only on downward crossing of active regime levels.
        downward_start = (
            open_price
            if previous_close is None
            else max(previous_close, open_price)
        )

        active_buy_prices = regime["buy_prices"]
        active_buy_price_list = active_buy_prices.tolist()

        if low_price < downward_start:
            first_index = bisect.bisect_left(
                active_buy_price_list,
                low_price,
            )
            stop_index = bisect.bisect_left(
                active_buy_price_list,
                downward_start,
            )

            for k in range(stop_index - 1, first_index - 1, -1):
                grid_buy_price = float(active_buy_prices[k])

                # Old positions survive re-centering. An old open position
                # blocks another position at the same logical grid price.
                if (
                    grid_buy_price in open_by_grid_price
                    or grid_buy_price in sold_this_candle
                ):
                    continue

                if risk_halt:
                    blocked["risk_halt"] += 1
                    break

                cost = normal_order_size

                if buy_budget + tolerance < cost:
                    break

                if (
                    buy_budget - cost
                    < min_cash_reserve - tolerance
                ):
                    blocked["cash_reserve"] += 1
                    break

                if (
                    max_open_positions is not None
                    and len(open_by_grid_price) >= max_open_positions
                ):
                    blocked["max_open_positions"] += 1
                    break

                if (
                    max_deployed_capital is not None
                    and deployed_capital + cost
                    > max_deployed_capital + tolerance
                ):
                    blocked["max_deployed_capital"] += 1
                    break

                gross_btc = cost / grid_buy_price
                projected_btc = (
                    btc + gross_btc * (1.0 - buy_fee)
                )
                projected_entry_exposure = (
                    projected_btc * grid_buy_price
                )

                if (
                    max_entry_btc_exposure is not None
                    and projected_entry_exposure
                    > max_entry_btc_exposure + tolerance
                ):
                    blocked["max_entry_btc_exposure"] += 1
                    break

                portfolio_value = cash + btc * grid_buy_price
                trade_id += 1

                position = create_position(
                    trade_id=trade_id,
                    regime_id=regime["regime_id"],
                    reference_price=regime["reference_price"],
                    buy_time=timestamp,
                    market_buy_price=grid_buy_price,
                    grid_buy_price=grid_buy_price,
                    sell_target=grid_buy_price + gap,
                    normal_order_size=normal_order_size,
                    portfolio_value_at_buy=portfolio_value,
                    buy_fee=buy_fee,
                    sell_fee=sell_fee,
                    entry_type="GRID_BUY",
                )

                cash_before, btc_before = cash, btc

                buy_budget -= cost
                cash -= cost
                btc += position["btc_amount"]
                deployed_capital += cost
                total_buy_fee_usdt += (
                    position["buy_fee_btc"] * grid_buy_price
                )

                positions[trade_id] = position
                open_by_grid_price[grid_buy_price] = trade_id
                heapq.heappush(
                    sell_heap,
                    (position["sell_target"], trade_id),
                )

                event_id += 1
                events.append({
                    "event_id": event_id,
                    "time": timestamp,
                    "side": "BUY",
                    "trade_id": trade_id,
                    "regime_id": regime["regime_id"],
                    "grid_buy_price": grid_buy_price,
                    "price": grid_buy_price,
                    "cash_movement": -cost,
                    "grid_cashflow": 0.0,
                    "cash_before": cash_before,
                    "cash_after": cash,
                    "btc_before": btc_before,
                    "btc_after": btc,
                    "deployed_capital_after": deployed_capital,
                    "entry_exposure_after": projected_entry_exposure,
                    "initialization_trade": False,
                })

        # 3) End-of-candle equity and drawdown.
        equity = cash + btc * close_price
        peak_equity = max(peak_equity, equity)
        current_drawdown = equity / peak_equity - 1.0

        # Drawdown halt is one-way for the remainder of this run.
        # It stops NEW BUYs only; existing positions can still SELL normally.
        if (
            not risk_halt
            and max_drawdown_stop is not None
            and current_drawdown <= -max_drawdown_stop
        ):
            risk_halt = True
            risk_halt_events.append({
                "decision_time": timestamp,
                "effective_time": (
                    data.iloc[i + 1]["open_time"]
                    if i + 1 < n
                    else pd.NaT
                ),
                "drawdown": float(current_drawdown),
                "equity": float(equity),
                "peak_equity": float(peak_equity),
            })

        equity_values[i] = equity
        cash_values[i] = cash
        btc_values[i] = btc
        deployed_values[i] = deployed_capital
        open_position_values[i] = len(open_by_grid_price)
        btc_market_value_values[i] = btc * close_price
        reference_values[i] = regime["reference_price"]
        regime_values[i] = regime["regime_id"]
        risk_halt_values[i] = risk_halt

        # 4) Re-center decision uses candle CLOSE and becomes effective
        # on the NEXT candle. Existing positions are untouched.
        trigger_distance = recenter_trigger_grids * gap
        if (
            close_price
            >= regime["reference_price"] + trigger_distance
            or close_price
            <= regime["reference_price"] - trigger_distance
        ):
            new_reference = round_to_gap(close_price, gap)

            if new_reference != regime["reference_price"]:
                old_regime = regime
                regime_id += 1
                regime = build_regime(
                    new_reference,
                    gap,
                    number_of_grids,
                    regime_id,
                )

                direction = (
                    "UP"
                    if new_reference > old_regime["reference_price"]
                    else "DOWN"
                )

                event = {
                    "decision_time": timestamp,
                    "effective_time": (
                        data.iloc[i + 1]["open_time"]
                        if i + 1 < n
                        else pd.NaT
                    ),
                    "direction": direction,
                    "close": close_price,
                    "old_reference": old_regime["reference_price"],
                    "new_reference": new_reference,
                    "old_floor": old_regime["floor"],
                    "old_ceiling": old_regime["ceiling"],
                    "new_floor": regime["floor"],
                    "new_ceiling": regime["ceiling"],
                }
                recenter_events.append(event)

                regime_history.append({
                    "regime_id": regime_id,
                    "effective_time": event["effective_time"],
                    "reference_price": regime["reference_price"],
                    "floor": regime["floor"],
                    "ceiling": regime["ceiling"],
                    "reason": f"RECENTER_{direction}",
                })

        previous_close = close_price

    equity_curve = pd.DataFrame({
        "open_time": data["open_time"],
        "close": data["close"],
        "cash": cash_values,
        "btc": btc_values,
        "btc_market_value": btc_market_value_values,
        "deployed_capital": deployed_values,
        "open_positions": open_position_values,
        "equity": equity_values,
        "reference_price": reference_values,
        "regime_id": regime_values,
        "risk_halt": risk_halt_values,
    })

    return {
        "data": data,
        "equity_curve": equity_curve,
        "positions": positions,
        "events": pd.DataFrame(events),
        "initialization": initialization,
        "regime_history": regime_history,
        "recenter_events": recenter_events,
        "risk_halt_events": risk_halt_events,
        "blocked_entries": blocked,
        "completed_cycles": completed_cycles,
        "realized_profit": float(realized_profit),
        "total_buy_fee_usdt": float(total_buy_fee_usdt),
        "total_sell_fee_usdt": float(total_sell_fee_usdt),
        "normal_order_size_usdt": normal_order_size,
        "final_cash": float(cash),
        "final_btc": float(btc),
        "final_deployed_capital": float(deployed_capital),
        "risk_halt": bool(risk_halt),
    }


# 2. Backtest System


## 2.1 Backtest Configuration & Data


In [ ]:
DATA_DIR = "/content/drive/MyDrive/03.Trading/00.Live Trading"
TIMEFRAME = "1m"
START_DATE = "2024-01-01"
END_DATE = "2026-01-01"


def load_market_data(symbol, timeframe, data_dir, start_date, end_date):
    path = os.path.join(
        data_dir,
        f"{symbol}-{timeframe}-combined.csv",
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    required = {
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
    }
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(
            f"Missing columns: {sorted(missing)}"
        )

    df["open_time"] = pd.to_datetime(
        df["open_time"],
        utc=True,
    )

    numeric = [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]
    df[numeric] = df[numeric].astype(float)

    start_ts = pd.Timestamp(start_date, tz="UTC")
    end_ts = pd.Timestamp(end_date, tz="UTC")

    return (
        df.drop_duplicates("open_time")
        .sort_values("open_time")
        .loc[
            lambda x:
            (x["open_time"] >= start_ts)
            & (x["open_time"] < end_ts)
        ]
        .reset_index(drop=True)
    )


## 2.2 Performance Metrics & BTC Buy/Hold Benchmark


In [ ]:
def performance_stats(data, equity, initial_capital):
    equity = np.asarray(equity, dtype=float)
    running_peak = np.maximum.accumulate(equity)
    drawdown = equity / running_peak - 1.0

    final_equity = float(equity[-1])
    net_return = final_equity / initial_capital - 1.0
    max_drawdown = float(drawdown.min())

    elapsed_days = (
        data["open_time"].iloc[-1]
        - data["open_time"].iloc[0]
    ).total_seconds() / 86400.0

    annualized_return = np.nan
    if elapsed_days > 0 and final_equity > 0:
        growth = (
            np.log(final_equity / initial_capital)
            * (365.25 / elapsed_days)
        )
        if growth < 700:
            annualized_return = float(np.expm1(growth))

    calmar_ratio = np.nan
    if (
        max_drawdown < 0
        and np.isfinite(annualized_return)
    ):
        calmar_ratio = float(
            annualized_return / abs(max_drawdown)
        )

    return {
        "final_equity": final_equity,
        "net_return": float(net_return),
        "annualized_return": annualized_return,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar_ratio,
        "drawdown": drawdown,
    }


def build_buy_hold_benchmark(
    data,
    initial_capital,
    buy_fee,
):
    entry_price = float(data.iloc[0]["open"])
    final_price = float(data.iloc[-1]["close"])

    gross_btc = initial_capital / entry_price
    net_btc = gross_btc * (1.0 - buy_fee)
    entry_fee_usdt_equiv = (
        gross_btc * buy_fee * entry_price
    )

    equity = net_btc * data["close"].to_numpy(float)
    stats = performance_stats(
        data,
        equity,
        initial_capital,
    )

    return {
        "entry_price": entry_price,
        "final_price": final_price,
        "gross_btc": float(gross_btc),
        "net_btc": float(net_btc),
        "entry_fee_usdt_equiv": float(
            entry_fee_usdt_equiv
        ),
        "final_equity": stats["final_equity"],
        "net_return": stats["net_return"],
        "annualized_return": stats["annualized_return"],
        "max_drawdown": stats["max_drawdown"],
        "calmar_ratio": stats["calmar_ratio"],
    }


## 2.3 Trade History & Audit


In [ ]:
def build_trade_history(
    positions,
    final_time,
    final_close,
):
    rows = []

    for trade_id in sorted(positions):
        p = positions[trade_id]

        if p["status"] == "CLOSED":
            net_pnl = float(p["net_pnl"])
            holding_time = p["sell_time"] - p["buy_time"]
        else:
            net_pnl = (
                p["btc_amount"] * final_close
                - p["order_size_usdt"]
            )
            holding_time = final_time - p["buy_time"]

        rows.append({
            "Trade ID": p["trade_id"],
            "Regime ID": p["regime_id"],
            "Entry Type": p["entry_type"],
            "Status": p["status"],
            "Buy Time": p["buy_time"],
            "Buy Price": p["buy_price"],
            "Grid Buy Price": p["grid_buy_price"],
            "Reference at Buy": p["reference_price_at_buy"],
            "Sell Target": p["sell_target"],
            "Sell Time": p["sell_time"],
            "Order Size (USDT)": p["order_size_usdt"],
            "Portfolio Value at Buy": p["portfolio_value_at_buy"],
            "Order % of Portfolio": p["order_pct_of_portfolio"],
            "Portfolio Value at Sell": p["portfolio_value_at_sell"],
            "Net P&L": net_pnl,
            "Holding Time": holding_time,
        })

    return pd.DataFrame(rows)


def audit_v1(
    result,
    initial_capital,
    max_open_positions,
    max_deployed_capital,
    max_entry_btc_exposure,
):
    positions = result["positions"]
    events = result["events"]
    curve = result["equity_curve"]
    initialization = result["initialization"]

    open_positions = [
        p for p in positions.values()
        if p["status"] == "OPEN"
    ]
    closed_positions = [
        p for p in positions.values()
        if p["status"] == "CLOSED"
    ]

    final_close = float(result["data"].iloc[-1]["close"])
    open_btc = sum(p["btc_amount"] for p in open_positions)
    open_cost = sum(
        p["order_size_usdt"] for p in open_positions
    )

    final_equity_identity = (
        result["final_cash"]
        + result["final_btc"] * final_close
    )

    initial_reconciliation = (
        initialization["initial_cash"]
        + initialization["initial_btc_cost_usdt"]
    )

    non_seed_costs = [
        p["order_size_usdt"]
        for p in positions.values()
        if p["entry_type"] == "GRID_BUY"
    ]
    fixed_order_size_ok = all(
        np.isclose(
            c,
            result["normal_order_size_usdt"],
            atol=1e-9,
        )
        for c in non_seed_costs
    )

    # Check that initialization seed quantity equals a normal future BUY
    # quantity for the same logical grid level.
    seed_quantity_ok = True
    for p in positions.values():
        if p["entry_type"] != "INITIAL_SEED":
            continue
        expected_btc = (
            result["normal_order_size_usdt"]
            / p["grid_buy_price"]
            * (1.0 - BUY_FEE)
        )
        if not np.isclose(
            p["btc_amount"],
            expected_btc,
            atol=1e-12,
        ):
            seed_quantity_ok = False
            break

    same_candle_rebuy_ok = True
    if len(events):
        sell_events = events.loc[
            events["side"].eq("SELL"),
            ["time", "grid_buy_price"],
        ]
        buy_events = events.loc[
            events["side"].eq("BUY")
            & ~events["initialization_trade"].fillna(False),
            ["time", "grid_buy_price"],
        ]
        if len(sell_events) and len(buy_events):
            overlap = sell_events.merge(
                buy_events,
                on=["time", "grid_buy_price"],
                how="inner",
            )
            same_candle_rebuy_ok = overlap.empty

    max_open_observed = int(curve["open_positions"].max())
    max_deployed_observed = float(
        curve["deployed_capital"].max()
    )

    max_entry_exposure_observed = float(
        initialization["initial_btc_market_value_usdt"]
    )
    if (
        len(events)
        and "entry_exposure_after" in events.columns
    ):
        exposures = pd.to_numeric(
            events["entry_exposure_after"],
            errors="coerce",
        ).dropna()
        if len(exposures):
            max_entry_exposure_observed = max(
                max_entry_exposure_observed,
                float(exposures.max()),
            )

    checks = {
        "initial_capital_reconciliation": bool(
            np.isclose(
                initial_reconciliation,
                initial_capital,
                atol=1e-8,
            )
        ),
        "cash_never_negative": bool(
            (curve["cash"] >= -1e-8).all()
        ),
        "btc_never_negative": bool(
            (curve["btc"] >= -1e-12).all()
        ),
        "final_equity_identity": bool(
            np.isclose(
                curve["equity"].iloc[-1],
                final_equity_identity,
                atol=1e-8,
            )
        ),
        "final_btc_matches_open_positions": bool(
            np.isclose(
                result["final_btc"],
                open_btc,
                atol=1e-12,
            )
        ),
        "deployed_capital_matches_open_cost": bool(
            np.isclose(
                result["final_deployed_capital"],
                open_cost,
                atol=1e-8,
            )
        ),
        "closed_trade_count_reconciliation": bool(
            result["completed_cycles"]
            == len(closed_positions)
        ),
        "initial_seed_quantity_matches_normal_grid": bool(
            seed_quantity_ok
        ),
        "fixed_order_size_no_compounding": bool(
            fixed_order_size_ok
        ),
        "no_same_candle_sell_rebuy": bool(
            same_candle_rebuy_ok
        ),
        "max_open_positions_respected": bool(
            max_open_positions is None
            or max_open_observed <= max_open_positions
        ),
        "max_deployed_capital_respected": bool(
            max_deployed_capital is None
            or max_deployed_observed
            <= max_deployed_capital + 1e-8
        ),
        "max_entry_btc_exposure_respected": bool(
            max_entry_btc_exposure is None
            or max_entry_exposure_observed
            <= max_entry_btc_exposure + 1e-8
        ),
    }

    return {
        "status": "PASS"
        if all(checks.values())
        else "FAIL",
        "checks": checks,
        "diagnostics": {
            "max_open_positions_observed": max_open_observed,
            "max_deployed_capital_observed": max_deployed_observed,
            "max_entry_btc_exposure_observed": (
                max_entry_exposure_observed
            ),
        },
    }


## 2.4 Run Backtest & Review Results


In [ ]:
data = load_market_data(
    SYMBOL,
    TIMEFRAME,
    DATA_DIR,
    START_DATE,
    END_DATE,
)

result = run_dynamic_recenter_grid(
    data=data,
    initial_capital=INITIAL_CAPITAL,
    initial_reference=INITIAL_REFERENCE,
    gap=GRID_GAP,
    number_of_grids=NUMBER_OF_GRIDS,
    recenter_trigger_grids=RECENTER_TRIGGER_GRIDS,
    buy_fee=BUY_FEE,
    sell_fee=SELL_FEE,
    min_cash_reserve=MIN_CASH_RESERVE,
    max_open_positions=MAX_OPEN_POSITIONS,
    max_deployed_capital=MAX_DEPLOYED_CAPITAL,
    max_entry_btc_exposure=MAX_ENTRY_BTC_EXPOSURE,
    max_drawdown_stop=MAX_DRAWDOWN_STOP,
)

v1_stats = performance_stats(
    result["data"],
    result["equity_curve"]["equity"].to_numpy(float),
    INITIAL_CAPITAL,
)
result["equity_curve"]["drawdown"] = v1_stats["drawdown"]

buy_hold = build_buy_hold_benchmark(
    result["data"],
    INITIAL_CAPITAL,
    BUY_FEE,
)

final_time = result["data"].iloc[-1]["open_time"]
final_close = float(result["data"].iloc[-1]["close"])
trade_history = build_trade_history(
    result["positions"],
    final_time,
    final_close,
)

open_positions = [
    p for p in result["positions"].values()
    if p["status"] == "OPEN"
]
unrealized_pnl = float(sum(
    p["btc_amount"] * final_close
    - p["order_size_usdt"]
    for p in open_positions
))

audit = audit_v1(
    result=result,
    initial_capital=INITIAL_CAPITAL,
    max_open_positions=MAX_OPEN_POSITIONS,
    max_deployed_capital=MAX_DEPLOYED_CAPITAL,
    max_entry_btc_exposure=MAX_ENTRY_BTC_EXPOSURE,
)

summary = {
    "initial_capital": INITIAL_CAPITAL,
    "final_equity": v1_stats["final_equity"],
    "net_return": v1_stats["net_return"],
    "annualized_return": v1_stats["annualized_return"],
    "max_drawdown": v1_stats["max_drawdown"],
    "calmar_ratio": v1_stats["calmar_ratio"],
    "completed_cycles": result["completed_cycles"],
    "open_positions": len(open_positions),
    "final_cash": result["final_cash"],
    "final_btc": result["final_btc"],
    "final_deployed_capital": result["final_deployed_capital"],
    "realized_profit": result["realized_profit"],
    "unrealized_pnl": unrealized_pnl,
    "total_fee_usdt_equiv": (
        result["total_buy_fee_usdt"]
        + result["total_sell_fee_usdt"]
    ),
    "recenter_count": len(result["recenter_events"]),
    "risk_halt": result["risk_halt"],
    "blocked_entries": result["blocked_entries"],
}

comparison_vs_buy_hold = {
    "final_equity_difference_usdt": (
        summary["final_equity"]
        - buy_hold["final_equity"]
    ),
    "excess_return": (
        summary["net_return"]
        - buy_hold["net_return"]
    ),
    "drawdown_improvement": (
        abs(buy_hold["max_drawdown"])
        - abs(summary["max_drawdown"])
    ),
    "calmar_difference": (
        summary["calmar_ratio"]
        - buy_hold["calmar_ratio"]
    ),
}

print("=== V1 Dynamic Re-centering Grid ===")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n=== Initialization ===")
for key, value in result["initialization"].items():
    print(f"{key}: {value}")

print("\n=== BTC Buy & Hold ===")
for key, value in buy_hold.items():
    print(f"{key}: {value}")

print("\n=== V1 vs Buy & Hold ===")
for key, value in comparison_vs_buy_hold.items():
    print(f"{key}: {value}")

print("\n=== Audit ===")
print(audit["status"])
for key, value in audit["checks"].items():
    print(f"{key}: {value}")

print("\n=== Audit Diagnostics ===")
for key, value in audit["diagnostics"].items():
    print(f"{key}: {value}")

print("\n=== Re-centering ===")
print(f"Count: {len(result['recenter_events'])}")
if result["recenter_events"]:
    display(pd.DataFrame(result["recenter_events"]).head(20))


## 2.5 User Trade History


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

display(trade_history.head(100))
print(f"Total trades: {len(trade_history)}")
print(f"Closed      : {(trade_history['Status'] == 'CLOSED').sum()}")
print(f"Open        : {(trade_history['Status'] == 'OPEN').sum()}")


# 3. Logging System

V1 uses immutable run folders, matching the safer V0 logging architecture:

`logs/v1/<run_id>/summary.json`

`logs/v1/<run_id>/trade_history.csv`

A new run never overwrites a previous result.


## 3.1 Build Immutable Run Log


In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]

    if isinstance(value, np.ndarray):
        return [json_safe(v) for v in value.tolist()]

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        value = float(value)
        return value if np.isfinite(value) else None

    if isinstance(value, (pd.Timestamp, datetime)):
        if pd.isna(value):
            return None
        return value.isoformat()

    if isinstance(value, pd.Timedelta):
        return str(value)

    if value is pd.NaT:
        return None

    if isinstance(value, float):
        return value if np.isfinite(value) else None

    if isinstance(value, (bool, str, int)) or value is None:
        return value

    return str(value)


RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)
RUN_DIR = f"logs/v1/{RUN_ID}"
SUMMARY_PATH = f"{RUN_DIR}/summary.json"
TRADE_HISTORY_PATH = f"{RUN_DIR}/trade_history.csv"

log_payload = {
    "log_schema_version": 3,
    "run_info": {
        "run_id": RUN_ID,
        "generated_at": datetime.now(
            timezone.utc
        ).isoformat(),
        "strategy": "V1 Dynamic Re-centering Grid",
        "execution_mode": EXECUTION_MODE,
        "live_execution_enabled": LIVE_EXECUTION_ENABLED,
    },
    "trading_config": {
        "symbol": SYMBOL,
        "initial_capital": INITIAL_CAPITAL,
        "initial_floor": INITIAL_FLOOR,
        "initial_ceiling": INITIAL_CEILING,
        "initial_reference": INITIAL_REFERENCE,
        "grid_gap": GRID_GAP,
        "number_of_grids": NUMBER_OF_GRIDS,
        "recenter_trigger_grids": RECENTER_TRIGGER_GRIDS,
        "recenter_trigger_usdt": (
            RECENTER_TRIGGER_GRIDS * GRID_GAP
        ),
        "buy_fee": BUY_FEE,
        "sell_fee": SELL_FEE,
        "normal_order_size_usdt": (
            result["normal_order_size_usdt"]
        ),
        "no_compounding": True,
        "old_positions_survive_recenter": True,
    },
    "risk_guardrails": {
        "min_cash_reserve": MIN_CASH_RESERVE,
        "max_open_positions": MAX_OPEN_POSITIONS,
        "max_deployed_capital": MAX_DEPLOYED_CAPITAL,
        "max_entry_btc_exposure": MAX_ENTRY_BTC_EXPOSURE,
        "max_drawdown_stop": MAX_DRAWDOWN_STOP,
        "drawdown_halt_behavior": (
            "one-way halt of NEW BUY entries; "
            "existing positions may still SELL"
        ),
    },
    "backtest_config": {
        "timeframe": TIMEFRAME,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "data_rows": len(result["data"]),
        "first_candle": result["data"].iloc[0]["open_time"],
        "last_candle": result["data"].iloc[-1]["open_time"],
    },
    "initialization": result["initialization"],
    "summary": summary,
    "buy_hold_benchmark": buy_hold,
    "comparison_vs_buy_hold": comparison_vs_buy_hold,
    "recenter_events": result["recenter_events"],
    "risk_halt_events": result["risk_halt_events"],
    "audit": audit,
    "trade_history_file": TRADE_HISTORY_PATH,
}

log_payload = json_safe(log_payload)

print(f"Run ID: {RUN_ID}")
print(f"Summary: {SUMMARY_PATH}")
print(f"Trade history: {TRADE_HISTORY_PATH}")


## 3.2 Upload Immutable Run Logs to GitHub


In [ ]:
from google.colab import userdata


def create_github_file(
    repo,
    branch,
    path,
    text_content,
    token,
    message,
    max_retries=3,
):
    api_url = (
        f"https://api.github.com/repos/{repo}/contents/{path}"
    )
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    # Immutable create-only behavior.
    existing = requests.get(
        api_url,
        headers=headers,
        params={"ref": branch},
        timeout=30,
    )

    if existing.status_code == 200:
        raise FileExistsError(
            f"Immutable log path already exists: {path}"
        )
    if existing.status_code != 404:
        existing.raise_for_status()

    body = {
        "message": message,
        "content": base64.b64encode(
            text_content.encode("utf-8")
        ).decode("ascii"),
        "branch": branch,
    }

    for attempt in range(1, max_retries + 1):
        response = requests.put(
            api_url,
            headers=headers,
            json=body,
            timeout=30,
        )

        if response.status_code in (200, 201):
            payload = response.json()
            return {
                "commit_sha": payload["commit"]["sha"],
                "content_sha": payload["content"]["sha"],
                "path": path,
            }

        if (
            response.status_code == 409
            and attempt < max_retries
        ):
            time.sleep(attempt)
            continue

        response.raise_for_status()


github_token = userdata.get("GITHUB_TOKEN")

summary_upload = create_github_file(
    repo="natdanaiii/Trading",
    branch="main",
    path=SUMMARY_PATH,
    text_content=json.dumps(
        log_payload,
        indent=2,
        allow_nan=False,
    ),
    token=github_token,
    message=f"Add V1 backtest summary {RUN_ID}",
)

trade_history_upload = create_github_file(
    repo="natdanaiii/Trading",
    branch="main",
    path=TRADE_HISTORY_PATH,
    text_content=trade_history.to_csv(index=False),
    token=github_token,
    message=f"Add V1 trade history {RUN_ID}",
)

print("GitHub immutable log upload: SUCCESS")
print(summary_upload)
print(trade_history_upload)
